In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
!pip install lftk spacy
!python -m spacy download pt_core_news_sm

In [ ]:
import re
import os
import gc
import math
import lftk
import spacy

import pandas as pd

from tqdm import tqdm

In [ ]:
def limpar_texto_para_lftk(texto):
    if not isinstance(texto, str):
        return ""
    texto = re.sub(r'http\S+|www\S+|https\S+', ' ', texto)
    texto = re.sub(r'<.*?>', ' ', texto)
    texto = re.sub(r'@\w+', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

caminho_original = "CAMINHO DO ARQUIVO BASE"
df = pd.read_csv(caminho_original)

df['textClean'] = df['commentText'].apply(limpar_texto_para_lftk)
df = df[df['textClean'] != ""]

df_lftk = df[['commentText', 'textClean', 'feeling']].copy()
df_lftk.reset_index(drop=True, inplace=True)

caminho_novo = "SAÍDA DO ARQUIVO LIMPO"
df_lftk.to_csv(caminho_novo, index=False)

print(f"O arquivo limpo foi salvo como: {caminho_novo}")

In [ ]:
caminho_entrada = "CAMINHO DO ARQUIVO BASE"
caminho_saida = "CAMINHO DO ARQUIVO DE SAÍDA"

LINHA_INICIAL = 0 # Linha inicial onde o lftk vai começar
LINHA_FINAL = 140 # Linha onde ele vai terminar
tamanho_lote = 1000

print("Carregando o modelo gramatical do spaCy (sm)...")
nlp = spacy.load("pt_core_news_sm")

print("Carregando a base de dados limpa...")
df_completo = pd.read_csv(caminho_entrada)
coluna_texto = 'textClean'
df_completo.dropna(subset=[coluna_texto], inplace=True)

tamanho_maximo = 100000 # Filtro para textos grandes
df_completo = df_completo[df_completo[coluna_texto].str.len() <= tamanho_maximo]
df_completo.reset_index(drop=True, inplace=True)

total_linhas_reais = len(df_completo)
LINHA_FINAL = min(LINHA_FINAL, total_linhas_reais)
linhas_nesta_maquina = LINHA_FINAL - LINHA_INICIAL
total_lotes_nesta_maquina = math.ceil(linhas_nesta_maquina / tamanho_lote)

print(f"Esta máquina vai processar da linha {LINHA_INICIAL} até {LINHA_FINAL}.")
print(f"Total de Lotes: {total_lotes_nesta_maquina}\n")

for inicio in range(LINHA_INICIAL, LINHA_FINAL, tamanho_lote):
    fim = min(inicio + tamanho_lote, LINHA_FINAL)
    lote_relativo = ((inicio - LINHA_INICIAL) // tamanho_lote) + 1

    print(f"Iniciando Lote {lote_relativo} de {total_lotes_nesta_maquina}...")

    df_lote = df_completo.iloc[inicio:fim].copy()
    textos = df_lote[coluna_texto].astype(str).tolist()

    doc_generator = nlp.pipe(textos, batch_size=256, disable=["ner", "textcat"])
    lote_features = []

    for doc in tqdm(doc_generator, total=len(textos), desc="Analisando"):
        extrator = lftk.Extractor(docs=doc)
        features = extrator.extract()
        lote_features.append(features)

    df_features = pd.DataFrame(lote_features)
    df_lote_final = pd.concat([df_lote.reset_index(drop=True), df_features.reset_index(drop=True)], axis=1)

    modo_escrita = 'w' if inicio == LINHA_INICIAL else 'a'
    escrever_cabecalho = True if inicio == LINHA_INICIAL else False

    df_lote_final.to_csv(caminho_saida, mode=modo_escrita, header=escrever_cabecalho, index=False)
    print(f"Lote {lote_relativo} salvo no Drive!\n")

    del lote_features, df_features, df_lote_final, df_lote, textos

    gc.collect()

print(f"\nExtração concluída! Arquivo salvo em: {caminho_saida}")